O notebook - Conformance Checking — é a Camada 2 do framework.

Enquanto o notebook 1 verifica se cada requisito normativo R1-R7 está presente nos documentos do PADO, o notebook 2 verifica se o fluxo processual seguiu a sequência correta.
São perguntas diferentes:

- Notebook 1 pergunta: "O documento de notificação existe?"
- Notebook 2 pergunta: "A notificação aconteceu depois do despacho de instauração, na ordem correta?"


O notebook 2 faz isso em 4 perspectivas usando pm4py:

- Perspectiva 1 — Fluxo: a sequência A1, A2, A3, A4 foi respeitada?
- Perspectiva 2 — Dados: todas as atividades obrigatórias estão presentes?
- Perspectiva 3 — Recursos: os documentos têm responsável identificado?
- Perspectiva 4 — Tempo: os prazos entre as etapas foram respeitados?

In [2]:
# 1.  Imports e configuração
import re
import fitz
import pandas as pd
import pm4py
from pathlib import Path
from datetime import datetime
from pm4py.algo.discovery.alpha import algorithm as alpha_miner
from pm4py.algo.conformance.tokenreplay import algorithm as token_replay

# Mesmo caminho do notebook de produção
PASTA_PADOS = r"D:\WULDSON\Estudos\Mestrado\IFPB\Disciplinas\orientacao\Transparencia_IA\projeto_anatel\dados\pdfs_anatel"
ARQUIVO_EVENTLOG = r"event_log_instauracao.csv"

print(f"Imports carregados")
print(f"Pasta PADOs: {Path(PASTA_PADOS).resolve()}")
print(f"pm4py versão: {pm4py.__version__}")

Imports carregados
Pasta PADOs: D:\WULDSON\Estudos\Mestrado\IFPB\Disciplinas\orientacao\Transparencia_IA\projeto_anatel\dados\pdfs_anatel
pm4py versão: 2.7.20.1


### funções de extração de texto e classificação de documento para montar o event log sem depender do outro notebook estar aberto

In [3]:
# Carrega matriz de conformidade do notebook de produção
# para obter o Tipo_PADO de cada processo
import pandas as pd

ARQUIVO_MATRIZ = r"resultado_requisitos_pados_sf.xlsx"

df_matriz = pd.read_excel(ARQUIVO_MATRIZ, sheet_name='Matriz de Conformidade')
tipo_pado = dict(zip(df_matriz['PADO'], df_matriz['Tipo_PADO']))

print(f"Matriz carregada — {len(tipo_pado)} PADOs")
for k, v in tipo_pado.items():
    print(f"   {k} -> {v}")



# 2. Funções base (reaproveitadas do notebook de produção)
def extrair_texto_pdf(caminho_pdf):
    try:
        doc = fitz.open(str(caminho_pdf))
        texto = ""
        for pagina in doc:
            texto += pagina.get_text()
        doc.close()
        texto = texto.replace('\xa0', ' ')
        texto = re.sub(r' +', ' ', texto)
        return texto
    except:
        return ""


def detectar_tipo_documento(texto):
    t = texto.lower()
    if re.search(r"despacho\s+ordin[aá]t[oó]rio\s+de\s+instaurac[aã]o", t):
        return "Despacho de Instauração"
    if re.search(r"despacho\s+decis[oó]rio", t):
        return "Despacho Decisório"
    if re.search(r"despacho\s+ordin[aá]t[oó]rio", t):
        return "Despacho Ordinatório"
    if re.search(r"informe\s+n[oº°]", t) and re.search(r"conclus[aã]o|proposta", t):
        return "Informe"
    if re.search(r"relat[oó]rio\s+de\s+fiscaliza[cç][aã]o", t):
        return "Relatório de Fiscalização"
    if re.search(r"auto\s+de\s+infra[cç][aã]o", t) and re.search(r"agente\s+de\s+fiscaliza[cç][aã]o", t):
        return "Auto de Infração"
    if re.search(r"tipo\s+de\s+intima[cç][aã]o", t):
        return "Certidão de Intimação"
    if re.search(r"certid[aã]o\s+de\s+redistribuic[aã]o", t):
        return "Certidão de Redistribuição"
    if re.search(r"certid[aã]o", t):
        return "Certidão"
    if re.search(r"[oó]ficio\s+n[oº°]", t) and re.search(r"intima|notifica|alegac|renuncia", t):
        return "Ofício de Notificação"
    if re.search(r"requerimento\s+de\s+informa[cç][oõ]es", t):
        return "Requerimento de Informações"
    if re.search(r"aviso\s+de\s+recebimento", t):
        return "Aviso de Recebimento (AR)"
    if re.search(r"objeto\s+entregue|rastreamento\s+do\s+objeto", t):
        return "Rastreamento Correios"
    if re.search(r"[oó]ficio\s+n[oº°]", t):
        return "Ofício"
    if re.search(r"recurso|peti[cç][aã]o", t):
        return "Petição/Recurso"
    if re.search(r"recibo\s+de\s+peticionamento|usu[aá]rio\s+externo\s+\(signat", t):
        return "Recibo de Peticionamento"
    if re.search(r"termo\s+de\s+(fiscaliza[cç][aã]o|qualifica[cç][aã]o)", t):
        return "Termo de Fiscalização"
    return "Outro"


# Mapeamento: tipo de documento -> atividade no event log
# Define quais tipos de documento representam eventos relevantes
# para a fase de Instauração

MAPA_ATIVIDADES = {
    "Auto de Infração":          "A1_Documento_Motivador",
    "Relatório de Fiscalização": "A1_Documento_Motivador",
    "Informe":                   "A2_Analise_Previa",
    "Despacho de Instauração":   "A3_Despacho_Instauracao",
    "Despacho Ordinatório":      "A3_Despacho_Instauracao",
    "Ofício de Notificação":     "A4_Notificacao_Autuado",
    "Certidão de Intimação":     "A4_Notificacao_Autuado",
    "Aviso de Recebimento (AR)": "A4_Notificacao_Autuado",
}

print("Funções base e mapeamento de atividades carregados")
print(f"   {len(MAPA_ATIVIDADES)} tipos de documento mapeados para atividades")

Matriz carregada — 104 PADOs
   53000.062585_2010-06 -> Tipo A (por Informe)
   53500.007555_2026-55 -> Indeterminado
   53500.008890_2026-71 -> Tipo A (por Informe)
   53500.008991_2025-61 -> Tipo A (por Informe)
   53500.008995_2025-49 -> Tipo A (por Informe)
   53500.009858_2026-11 -> Indeterminado
   53500.011614_2022-66 -> Tipo A (por Informe)
   53500.014139_2023-61 -> Tipo A (por Informe)
   53500.016622_2024-61 -> Tipo A (por Informe)
   53500.017655_2022-66 -> Tipo A (por Informe)
   53500.019425_2026-65 -> Tipo A (por Informe)
   53500.021979_2025-41 -> Indeterminado
   53500.022299_2024-64 -> Tipo A (por Informe)
   53500.030037_2023-92 -> Tipo A (por Informe)
   53500.034731_2025-41 -> Indeterminado
   53500.036351_2018-11 -> Tipo A (por Informe)
   53500.036487_2023-99 -> Tipo A (por Informe)
   53500.047569_2022-88 -> Tipo A (por Informe)
   53500.057042_2024-23 -> Tipo A (por Informe)
   53500.057071_2024-95 -> Tipo A (por Informe)
   53500.068455_2024-33 -> Indeterminad

In [4]:
# 3. Extração de data e responsável de cada PDF

def extrair_data(texto):
    """
    Extrai a data de assinatura do documento.
    Prioriza 'Assinado eletronicamente em DD/MM/AAAA' que é
    a data real do documento no SEI, não datas referenciadas no texto.
    """
    # Prioridade 1: data de assinatura eletrônica no SEI
    match = re.search(
        r'assinado\s+eletronicamente\s+em\s+(\d{1,2}/\d{1,2}/\d{4})',
        texto, re.IGNORECASE
    )
    if match:
        try:
            return datetime.strptime(match.group(1), "%d/%m/%Y")
        except:
            pass

    # Prioridade 2: "em DD/MM/AAAA HH:MM" — carimbo do SEI
    match = re.search(r'em\s+(\d{1,2}/\d{1,2}/\d{4})\s+\d{2}:\d{2}', texto)
    if match:
        try:
            return datetime.strptime(match.group(1), "%d/%m/%Y")
        except:
            pass

    # Prioridade 3: "em DD/MM/AAAA"
    match = re.search(r'em\s+(\d{1,2}/\d{1,2}/\d{4})', texto, re.IGNORECASE)
    if match:
        try:
            return datetime.strptime(match.group(1), "%d/%m/%Y")
        except:
            pass

    # Prioridade 4: "DD de mês de AAAA"
    meses = {
        'janeiro': '01', 'fevereiro': '02', 'março': '03', 'abril': '04',
        'maio': '05', 'junho': '06', 'julho': '07', 'agosto': '08',
        'setembro': '09', 'outubro': '10', 'novembro': '11', 'dezembro': '12'
    }
    match = re.search(
        r'(\d{1,2})\s+de\s+(janeiro|fevereiro|março|abril|maio|junho|julho|agosto|setembro|outubro|novembro|dezembro)\s+de\s+(\d{4})',
        texto, re.IGNORECASE
    )
    if match:
        try:
            dia = match.group(1)
            mes = meses[match.group(2).lower()]
            ano = match.group(3)
            return datetime.strptime(f"{dia}/{mes}/{ano}", "%d/%m/%Y")
        except:
            pass

    # Prioridade 5: qualquer DD/MM/AAAA — último recurso
    match = re.search(r'(\d{1,2}/\d{1,2}/\d{4})', texto)
    if match:
        try:
            return datetime.strptime(match.group(1), "%d/%m/%Y")
        except:
            pass

    return None


def extrair_responsavel(texto):
    """
    Extrai o nome do responsável pela assinatura do documento.
    Busca o padrão 'assinado eletronicamente por NOME, Cargo'
    """
    match = re.search(
        r'assinado\s+eletronicamente\s+por\s+([A-ZÀ-Ú][a-zA-ZÀ-ú\s]+),\s*([A-Za-zÀ-ú\s]+)',
        texto, re.IGNORECASE
    )
    if match:
        return match.group(1).strip()
    return "N/A"


def extrair_cargo(texto):
    """
    Extrai o cargo do responsável pela assinatura.
    """
    match = re.search(
        r'assinado\s+eletronicamente\s+por\s+[^,]+,\s*([^\n,]{5,80})',
        texto, re.IGNORECASE
    )
    if match:
        return match.group(1).strip()[:80]
    return "N/A"


# Testa extração no primeiro PADO
print("Teste de extração nos primeiros 5 documentos do PADO 0:\n")
pasta_raiz = Path(PASTA_PADOS)
subpastas = sorted([p for p in pasta_raiz.iterdir() if p.is_dir()])
pasta_teste = subpastas[0]

print(f" {pasta_teste.name}\n")
print(f"{'Arquivo':<35} {'Tipo':<28} {'Data':<12} {'Responsável'}")
print("─" * 100)

for pdf in sorted(pasta_teste.glob("*.pdf"))[:5]:
    texto = extrair_texto_pdf(pdf)
    tipo = detectar_tipo_documento(texto)
    data = extrair_data(texto)
    resp = extrair_responsavel(texto)
    data_str = data.strftime("%d/%m/%Y") if data else "—"
    print(f"{pdf.name:<35} {tipo:<28} {data_str:<12} {resp[:35]}")

Teste de extração nos primeiros 5 documentos do PADO 0:

 53000.062585_2010-06

Arquivo                             Tipo                         Data         Responsável
────────────────────────────────────────────────────────────────────────────────────────────────────
doc_12041966.pdf                    Despacho Ordinatório         29/05/2024   Amiraldo Salgado do Amaral
doc_12058621.pdf                    Despacho Ordinatório         29/05/2024   Amiraldo Salgado do Amaral
volume.pdf                          Despacho Ordinatório         25/09/1999   N/A


In [5]:
# 4. Monta o Event Log completo dos 16 PADOs

def montar_event_log_pado(pasta_pado):
    eventos = []
    arquivos = sorted(pasta_pado.glob("*.pdf"))
    nome_pasta = pasta_pado.name
    
    # Controla se já encontrou o primeiro Despacho Ordinatório
    ja_tem_despacho_instauracao = False

    for pdf in arquivos:
        texto = extrair_texto_pdf(pdf)
        tipo = detectar_tipo_documento(texto)

        if tipo not in MAPA_ATIVIDADES:
            continue

        # Despacho Ordinatório: só o primeiro é A3 (Instauração)
        # Os demais são despachos de andamento — ignorar
        if tipo == "Despacho Ordinatório":
            if ja_tem_despacho_instauracao:
                continue
            ja_tem_despacho_instauracao = True

        data = extrair_data(texto)
        responsavel = extrair_responsavel(texto)
        cargo = extrair_cargo(texto)
        atividade = MAPA_ATIVIDADES[tipo]

        if responsavel == "N/A":
            if re.search(r"certid[aã]o", texto.lower()):
                origem = "Sistema (gerado automaticamente)"
            else:
                origem = "Não identificado"
        else:
            origem = "Assinatura eletrônica"

        processo = nome_pasta.replace("_", ".")
        processo = re.sub(r'(\d{4})\.(\d{2})$', r'\1/\2', processo)

        eventos.append({
            "case_id":     processo,
            "activity":    atividade,
            "timestamp":   data,
            "doc_tipo":    tipo,
            "doc_arquivo": pdf.name,
            "recurso":     responsavel,
            "cargo":       cargo,
            "origem":      origem,
            "pado_pasta":  nome_pasta,
        })

    return eventos


# Processa todos os PADOs
print("Montando Event Log...\n")
todos_eventos = []

for pasta in subpastas:
    eventos = montar_event_log_pado(pasta)
    todos_eventos.extend(eventos)
    print(f"{pasta.name:<40} -> {len(eventos)} eventos")

# Monta DataFrame
df_log = pd.DataFrame(todos_eventos)

# Remove eventos sem data (não conseguiu extrair)
sem_data = df_log['timestamp'].isna().sum()
df_log = df_log.dropna(subset=['timestamp'])

print(f"\nEvent Log montado:")
print(f"   Total de eventos:        {len(df_log)}")
print(f"   Eventos sem data (removidos): {sem_data}")
print(f"   PADOs no log:            {df_log['case_id'].nunique()}")
print(f"\nAtividades encontradas:")
print(df_log['activity'].value_counts().to_string())

Montando Event Log...

53000.062585_2010-06                     -> 1 eventos
53500.007555_2026-55                     -> 2 eventos
53500.008890_2026-71                     -> 2 eventos
53500.008991_2025-61                     -> 5 eventos
53500.008995_2025-49                     -> 2 eventos
53500.009858_2026-11                     -> 1 eventos
53500.011614_2022-66                     -> 4 eventos
53500.014139_2023-61                     -> 3 eventos
53500.016622_2024-61                     -> 2 eventos
53500.017655_2022-66                     -> 4 eventos
53500.019425_2026-65                     -> 1 eventos
53500.021979_2025-41                     -> 4 eventos
53500.022299_2024-64                     -> 7 eventos
53500.030037_2023-92                     -> 5 eventos
53500.034731_2025-41                     -> 2 eventos
53500.036351_2018-11                     -> 16 eventos
53500.036487_2023-99                     -> 6 eventos
53500.047569_2022-88                     -> 11 eventos
535

In [9]:
# 5. Modelo de processo esperado da fase de Instauração
# Define a sequência normativa que todo PADO deve seguir
# Baseado no portal Cadeia de Valor da ANATEL e nos normativos
# Sequência esperada da fase de Instauração (modelo normativo)
# Tipo A: A1 -> A2 -> A3 -> A4
# Tipo B: A1 -> A3 -> A4 (sem A2 — Auto de Infração concentra R1+R3)

# Converte o event log para formato pm4py
#df_pm4py = df_log.copy()
#df_pm4py = df_pm4py.rename(columns={
#    "case_id":   "case:concept:name",
#    "activity":  "concept:name",
#    "timestamp": "time:timestamp",
#    "recurso":   "org:resource",
#})
#df_pm4py["time:timestamp"] = pd.to_datetime(df_pm4py["time:timestamp"])
#
#event_log = pm4py.convert_to_event_log(df_pm4py)
#print(f"Event log convertido para formato pm4py")
#print(f"   {len(event_log)} traces (PADOs)")
#
## Descobre o modelo de processo a partir dos dados reais (algoritmo Alpha)
#net, im, fm = alpha_miner.apply(event_log)
#print(f"Modelo descoberto pelo Alpha Miner")
#print(f"   Lugares: {len(net.places)}")
#print(f"   Transições: {len(net.transitions)}")
#
## Salva event log em CSV
#df_log.to_csv(ARQUIVO_EVENTLOG, index=False, sep=';', encoding='utf-8-sig')
#print(f"\nEvent log salvo em: {ARQUIVO_EVENTLOG}")

def verificar_fluxo_normativo(case_id, atividades, tipo):
    """
    Verifica se o PADO seguiu o fluxo normativo esperado.
    Retorna status, fitness e observacao.
    """
    tem_a1 = 'A1_Documento_Motivador' in atividades
    tem_a2 = 'A2_Analise_Previa' in atividades
    tem_a3 = 'A3_Despacho_Instauracao' in atividades
    tem_a4 = 'A4_Notificacao_Autuado' in atividades

    # Tipo A: exige A2, A3 e A4
    if "Tipo A" in tipo:
        if tem_a2 and tem_a3 and tem_a4:
            return "OK", 1.0, "Fluxo Tipo A completo: A2->A3->A4"
        elif tem_a3 and tem_a4:
            return "ATENCAO", 0.7, "A3 e A4 presentes mas A2 ausente"
        elif tem_a3 or tem_a4:
            return "ATENCAO", 0.5, f"Fluxo incompleto: A2={'SIM' if tem_a2 else 'NAO'} A3={'SIM' if tem_a3 else 'NAO'} A4={'SIM' if tem_a4 else 'NAO'}"
        else:
            return "ERRADO", 0.0, "Sem A3 e sem A4 — fluxo nao identificado"

    # Tipo B: exige A1, A3 e A4
    elif "Tipo B" in tipo:
        if tem_a1 and tem_a3 and tem_a4:
            return "OK", 1.0, "Fluxo Tipo B completo: A1->A3->A4"
        elif tem_a3 and tem_a4:
            return "ATENCAO", 0.7, "A3 e A4 presentes mas A1 ausente"
        elif tem_a3 or tem_a4:
            return "ATENCAO", 0.5, f"Fluxo incompleto: A1={'SIM' if tem_a1 else 'NAO'} A3={'SIM' if tem_a3 else 'NAO'} A4={'SIM' if tem_a4 else 'NAO'}"
        else:
            return "ERRADO", 0.0, "Sem A3 e sem A4 — fluxo nao identificado"

    # Indeterminado: verifica apenas A3 e A4
    else:
        if tem_a3 and tem_a4:
            return "OK", 1.0, "A3 e A4 presentes — fluxo minimo atendido"
        elif tem_a3 or tem_a4:
            return "ATENCAO", 0.5, f"Fluxo incompleto: A3={'SIM' if tem_a3 else 'NAO'} A4={'SIM' if tem_a4 else 'NAO'}"
        else:
            return "ERRADO", 0.0, "Sem A3 e sem A4"


# Agrupa atividades por PADO
from collections import defaultdict
atividades_por_pado = defaultdict(set)
for _, row in df_log.iterrows():
    atividades_por_pado[row['case_id']].add(row['activity'])

# Avalia cada PADO contra o modelo normativo
print("=" * 60)
print("PERSPECTIVA 1 — FLUXO (Modelo Normativo Manual)")
print("=" * 60)

fitness_scores = []
resultados_fluxo = []

for case_id, atividades in sorted(atividades_por_pado.items()):
    pado_key = re.sub(r'\.(\d{4}-\d{2})$', r'_\1', case_id)
    tipo = tipo_pado.get(pado_key, "Indeterminado")
    status, fitness, obs = verificar_fluxo_normativo(case_id, atividades, tipo)
    fitness_scores.append(fitness)
    resultados_fluxo.append({
        "case_id": case_id,
        "tipo": tipo,
        "status": status,
        "fitness": fitness,
        "obs": obs,
        "atividades": ", ".join(sorted(atividades))
    })
    print(f"  {status:<8} {case_id:<35} fitness={fitness:.2f} | {obs}")

media_fitness = sum(fitness_scores) / len(fitness_scores)
ok_count = sum(1 for r in resultados_fluxo if r['status'] == 'OK')
atencao_count = sum(1 for r in resultados_fluxo if r['status'] == 'ATENCAO')
errado_count = sum(1 for r in resultados_fluxo if r['status'] == 'ERRADO')

print(f"\n{'=' * 60}")
print(f"Fitness medio       : {media_fitness:.3f} ({media_fitness*100:.1f}%)")
print(f"PADOs OK            : {ok_count}/{len(resultados_fluxo)} ({ok_count/len(resultados_fluxo)*100:.1f}%)")
print(f"PADOs ATENCAO       : {atencao_count}/{len(resultados_fluxo)} ({atencao_count/len(resultados_fluxo)*100:.1f}%)")
print(f"PADOs ERRADO        : {errado_count}/{len(resultados_fluxo)} ({errado_count/len(resultados_fluxo)*100:.1f}%)")

PERSPECTIVA 1 — FLUXO (Modelo Normativo Manual)
  ATENCAO  53000.062585.2010-06                fitness=0.50 | Fluxo incompleto: A2=NAO A3=SIM A4=NAO
  ATENCAO  53500.007555.2026-55                fitness=0.50 | Fluxo incompleto: A3=SIM A4=NAO
  ATENCAO  53500.008890.2026-71                fitness=0.70 | A3 e A4 presentes mas A2 ausente
  ATENCAO  53500.008991.2025-61                fitness=0.70 | A3 e A4 presentes mas A2 ausente
  ATENCAO  53500.008995.2025-49                fitness=0.70 | A3 e A4 presentes mas A2 ausente
  ATENCAO  53500.009858.2026-11                fitness=0.50 | Fluxo incompleto: A3=SIM A4=NAO
  ATENCAO  53500.011614.2022-66                fitness=0.70 | A3 e A4 presentes mas A2 ausente
  ATENCAO  53500.014139.2023-61                fitness=0.70 | A3 e A4 presentes mas A2 ausente
  ATENCAO  53500.016622.2024-61                fitness=0.70 | A3 e A4 presentes mas A2 ausente
  OK       53500.017655.2022-66                fitness=1.00 | Fluxo Tipo A completo: A2->A3->

In [7]:
# 6. Conformance Checking (4 perspectivas)
# ── Perspectiva 1: FLUXO
# Verifica se a sequência de atividades segue o modelo normativo esperado

replayed_traces = token_replay.apply(event_log, net, im, fm)

print("=" * 60)
print("PERSPECTIVA 1 — FLUXO (Token Replay)")
print("=" * 60)

fitness_scores = []
for i, trace in enumerate(replayed_traces):
    case_id = event_log[i].attributes['concept:name']
    fitness = trace['trace_fitness']
    fitness_scores.append(fitness)
    status = "OK" if fitness >= 0.8 else "ATENCAO " if fitness >= 0.5 else "ERRADO"
    print(f"  {status} {case_id:<35} fitness={fitness:.3f}")

media_fitness = sum(fitness_scores) / len(fitness_scores)
print(f"\n  Fitness médio: {media_fitness:.3f} ({media_fitness*100:.1f}%)")


# ── Perspectiva 2: DADOS
# Verifica se os atributos dos documentos estão preenchidos corretamente

print("\n" + "=" * 60)
print("PERSPECTIVA 2 — DADOS")
print("=" * 60)

for case in event_log:
    case_id = case.attributes['concept:name']
    atividades = [e['concept:name'] for e in case]
    tem_a1 = 'A1_Documento_Motivador' in atividades
    tem_a2 = 'A2_Analise_Previa' in atividades
    tem_a3 = 'A3_Despacho_Instauracao' in atividades
    tem_a4 = 'A4_Notificacao_Autuado' in atividades
    completo = tem_a3 and tem_a4
    status = "OK" if completo else "ATENCAO "
    faltam = []
    if not tem_a1: faltam.append("A1")
    if not tem_a2: faltam.append("A2")
    if not tem_a3: faltam.append("A3")
    if not tem_a4: faltam.append("A4")
    falta_str = f"falta: {', '.join(faltam)}" if faltam else "completo"
    print(f"  {status} {case_id:<35} {falta_str}")


# ── Perspectiva 3: RECURSOS
# Verifica se os documentos têm responsável identificado

print("\n" + "=" * 60)
print("PERSPECTIVA 3 — RECURSOS")
print("=" * 60)

for case in event_log:
    case_id = case.attributes['concept:name']
    sem_responsavel = [
        e['concept:name'] for e in case
        if e.get('org:resource', 'N/A') == 'N/A'
        and e.get('origem', '') != 'Sistema (gerado automaticamente)'
    ]
    status = "OK" if not sem_responsavel else "ATENCAO "
    obs = f"sem responsável: {sem_responsavel}" if sem_responsavel else "todos identificados"
    print(f"  {status} {case_id:<35} {obs}")


# ── Perspectiva 4: TEMPO — 3 métricas
# RASA Art. 33: notificação deve ocorrer em sequência ao despacho

print("\n" + "=" * 60)
print("PERSPECTIVA 4 — TEMPO (3 métricas)")
print("=" * 60)

for case in event_log:
    case_id = case.attributes['concept:name']
    pado_key = re.sub(r'\.(\d{4}-\d{2})$', r'_\1', case_id)
    tipo = tipo_pado.get(pado_key, "Indeterminado")

    eventos_caso = {e['concept:name']: e['time:timestamp'] for e in case}
    t_a1 = eventos_caso.get('A1_Documento_Motivador')
    t_a2 = eventos_caso.get('A2_Analise_Previa')
    t_a3 = eventos_caso.get('A3_Despacho_Instauracao')
    t_a4 = eventos_caso.get('A4_Notificacao_Autuado')

    print(f"\n  PADO: {case_id} [{tipo}]")

    # Métrica 1 — A1 → A3: tempo para instaurar após documento motivador
    if t_a1 and t_a3:
        delta = (t_a3 - t_a1).days
        status = "OK" if delta <= 90 else "ATENCAO" if delta <= 180 else "CRITICO"
        print(f"     M1 A1->A3 (doc motivador -> despacho):  {status} {delta} dias")
    elif t_a2 and t_a3:
        delta = (t_a3 - t_a2).days
        status = "OK" if delta <= 90 else "ATENCAO" if delta <= 180 else "CRITICO"
        print(f"     M1 A2->A3 (analise previa -> despacho): {status} {delta} dias")
    else:
        print(f"     M1 A1/A2->A3: NAO CALCULAVEL — dados insuficientes")

    # Métrica 2 — A3 → A4: tempo para notificar após despacho
    if t_a3 and t_a4:
        delta = (t_a4 - t_a3).days
        if delta < 0:
            print(f"     M2 A3->A4 (despacho -> notificacao):  INCONSISTENCIA: A4 anterior ao A3 ({abs(delta)} dias)")
        else:
            status = "OK" if delta <= 30 else "ATENCAO" if delta <= 60 else "CRITICO"
            print(f"     M2 A3->A4 (despacho -> notificacao):  {status} {delta} dias")
    else:
        print(f"     M2 A3->A4: NAO CALCULAVEL — A3 ou A4 ausente")

    # Métrica 3 — total: duração completa da fase de Instauração
    t_inicio = t_a1 or t_a2 or t_a3
    t_fim = t_a4 or t_a3
    if t_inicio and t_fim and t_fim >= t_inicio:
        delta = (t_fim - t_inicio).days
        status = "OK" if delta <= 90 else "ATENCAO" if delta <= 180 else "CRITICO"
        print(f"     M3 total (inicio -> notificacao):       {status} {delta} dias")
    else:
        print(f"     M3 total: NAO CALCULAVEL — dados insuficientes")

c:\Users\WDS\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
replaying log with TBR, completed traces :: 100%|██████████| 57/57 [00:00<00:00, 2283.39it/s]

PERSPECTIVA 1 — FLUXO (Token Replay)
  OK 53000.062585.2010-06                fitness=1.000
  ATENCAO  53500.007555.2026-55                fitness=0.667
  ATENCAO  53500.008890.2026-71                fitness=0.667
  ERRADO 53500.008991.2025-61                fitness=0.333
  ATENCAO  53500.008995.2025-49                fitness=0.667
  OK 53500.009858.2026-11                fitness=1.000
  ERRADO 53500.011614.2022-66                fitness=0.400
  ATENCAO  53500.014139.2023-61                fitness=0.500
  ATENCAO  53500.016622.2024-61                fitness=0.667
  ERRADO 53500.017655.2022-66                fitness=0.400
  OK 53500.019425.2026-65                fitness=1.000
  ATENCAO  53500.021979.2025-41                fitness=0.500
  ERRADO 53500.022299.2024-64                fitness=0.250
  ERRADO 53500.030037.2023-92                fitness=0.333
  ATENCAO  53500.034731.2025-41                fitness=0.667
  ERRADO 53500.036351.2018-11                fitness=0.118
  ERRADO 53500.03

O fitness médio de 42.6% na Perspectiva 1 não significa que os PADOs estão errados — significa que o modelo descoberto pelo Alpha Miner é muito restritivo. O Alpha Miner espera uma sequência linear exata (A1→A2→A3→A4) e penaliza qualquer PADO que não tenha todas as 4 atividades nessa ordem.
Mas sabemos que:

Tipo A não tem A1 separado (o Informe é A2)

Tipo B não tem A2
A4 pode ter múltiplos eventos (AR + Ofício + Certidão)

Isso é esperado — não é problema do PADO, é o modelo que precisa ser refinado.

O que os resultados realmente mostram
Perspectiva 2 (Dados) — os achados mais confiáveis:

13/16 PADOs com A3 e A4 presentes — conformes
3 PADOs com problemas reais: 53504.006103, 53516.002345, 53542.000639 sem A4

Perspectiva 3 (Recursos) — achado interessante:

A4 sem responsável é esperado — AR dos Correios não tem assinatura eletrônica
A1 sem responsável nos PADOs legados e Tipo B é real

Perspectiva 4 (Tempo) — os achados mais relevantes:

2 PADOs com notificação ANTES do despacho — isso é inconsistência real ou problema de extração de data
Prazos longos (300+ dias) — achado substantivo para a dissertação

53500.083357.2023-45 — A Certidão de Intimação (A4) tem data de 15/09 mas o Despacho (A3) tem data de 29/09. Isso é problema de extração de data — a Certidão está capturando uma data anterior (provavelmente a data do Ofício que ela certifica, não a data da própria Certidão).

53524.000191.2018-02 — Todos os eventos têm data 18/01/2018 exceto o Despacho que é 20/04/2018. Isso é o processo legado — os documentos foram migrados do sistema RADAR para o SEI com datas originais, mas o Despacho foi gerado posteriormente no SEI. Inconsistência real do processo legado.